### 1.1.6. Unordered Sets (Sets of Strings)

This section covers **symbolic sets** — unordered sets whose members are character strings
rather than numbers. String sets are the natural index domain for entities whose identities
are names rather than counts or positions: products, cities, nutrients, intersections. The
section addresses quoting rules for string literals, the numeric-vs.-string type distinction,
the `symbolic` keyword on parameters that store string values, `.dat` file syntax for string
sets, amplpy patterns for supplying them from Python, and the interaction of string sets with
the binary set operators covered in section 1.1.4.

A string set is not a separate AMPL type — it is simply a set whose members happen to be
string values. All properties of unordered sets established in section 1.1.5 apply without
modification: no ordering, no position-dependent functions, `card()` and membership tests
valid, all five binary operators valid.

In [ ]:
# Syntax forms for sets of strings

# Form 1: declaration only — members come from a data file or amplpy
set SETNAME;

# Form 2: symbolic definition — members fixed in the model
set SETNAME = { "member1", "member2", ... };

# Form 3: with membership constraint
set SETNAME within PARENT_SET;

# Form 4: with default
set SETNAME default { "member1", "member2", ... };

# Symbolic parameter referencing a string set member
param paramname symbolic [ in SETNAME ] [ <> other_param ] [ default "value" ] ;

**Syntax placeholders:**

| Placeholder | Meaning |
|---|---|
| `SETNAME` | Declared name of the set; any sequence of letters, digits, and `_` that is not a reserved word and not a legal number |
| `"member1", ...` | String literals enclosed in single or double quotes; in model context quotes are mandatory |
| `PARENT_SET` | Any previously declared set; membership constraint validated at data-load time |
| `paramname` | Declared name of the parameter |
| `symbolic` | Keyword permitting the parameter to hold a string value |
| `in SETNAME` | Restricts the parameter's value to be a member of `SETNAME`; checked at data-load time |
| `<> other_param` | Inequality constraint: this parameter must differ from `other_param`; both must be `symbolic` |
| `default "value"` | Fallback value used when no data is supplied |

#### String Members: Quoting Rules

AMPL distinguishes two contexts for string literals: **model context** (inside `.mod` files and
`ampl.eval()` calls) and **data context** (inside `.dat` files and `data;` sections).

**Model context — quotes are always required.**
Any string literal inside a model expression must be enclosed in single quotes (`'A&P'`) or
double quotes (`"Bell+Howell"`). Both delimiters are equivalent; the choice is the modeler's.
A literal string that appears without quotes in a model expression is interpreted as a name
(a reference to a declared set, parameter, or other component), not as a string value.

**Data context — quotes are optional for simple strings.**
Inside a `data;` section or `.dat` file, AMPL allows the quotes around a string to be omitted
if the string consists only of characters that could appear in a name or number (letters, digits,
underscores, `.`, `+`, `-`), provided the string would not otherwise be parsed as a number.
The string `bands` may be written without quotes; the string `A&P` requires quotes because
`&` is not a valid name character.

**Case sensitivity.**
Upper-case and lower-case letters are always distinct in AMPL. The strings `"fish"`, `"Fish"`,
and `"FISH"` denote three different set members in every context.

**Numbers vs. numeric-looking strings.**
The number `1` and the string `"1"` are distinct AMPL objects. A number is stored as a
floating-point value; different representations of the same number (`100` and `1E+2`) are the
same set member. A string that looks like a number (e.g., `"100"`) is a character sequence,
not a floating-point value, and cannot participate in arithmetic. In a data context, writing
`100` (without quotes) produces the number 100; writing `"100"` produces the string `"100"`.
Mixing numbers and strings in the same set is syntactically valid but almost always a modelling
error.

In [ ]:
# --- MODEL CONTEXT: quotes are always required for string literals ---

# Correct: both quote styles are accepted
set SHIFTS = {"Day", "Evening", "Night"};      # double quotes
set SEASONS = {'Spring', 'Summer', 'Autumn', 'Winter'};  # single quotes

# Correct: string containing special characters must be quoted
set CODES = {'A&P', '23RPFG', '486/33C', 'New_York'};

# Membership test with a string literal — quotes required in model context
param has_day = if "Day" in SHIFTS then 1 else 0;

# Number 1 and string "1" are distinct
set MIXED = {1, "1"};        # two distinct members: numeric 1 and string "1"
display card(MIXED);         # 2 — not 1

#### Data Context: Quoting Rules in `.dat` Files

In a `data;` section or `.dat` file the rules relax slightly. Quoting remains mandatory for
strings that contain characters outside the set `[A-Za-z0-9_.+-]`. For alphanumeric strings
(letters, digits, underscores) that would not be interpreted as numbers, quotes are optional.

The table below summarises when quotes are needed.

| String value | In model context | In data context | Reason |
|---|---|---|---|
| `bands` | `"bands"` required | `bands` or `"bands"` | Alphanumeric — quotes optional in data |
| `New_York` | `"New_York"` required | `New_York` or `"New_York"` | Underscore is a valid name character |
| `A&P` | `"A&P"` required | `"A&P"` required | `&` is not a valid name character |
| `27sep` | `"27sep"` required | `27sep` or `"27sep"` | Starts with digit; not a legal number |
| `100` | `"100"` for the string; `100` for the number | `"100"` for the string; `100` for the number | Distinguish string from number |
| `Gary, Indiana` | `"Gary, Indiana"` required | `"Gary, Indiana"` required | Contains space and comma |

In [ ]:
# --- DATA CONTEXT examples (.dat file or data; section) ---

# Unquoted alphanumeric members are accepted in data context
data;
set PROD := bands coils plate ;          # quotes optional for simple identifiers

# Equivalent with quotes (also correct)
set PROD := "bands" "coils" "plate" ;    # quotes present but not required here

# Members with special characters require quotes even in data context
set CITIES := 'Gary, Indiana'  'Cleveland, Ohio'  'Pittsburgh, PA' ;

# Short codes that start with digits: no quotes needed (they are not legal numbers)
set WEEKS := 27sep 04oct 11oct 18oct ;

# Numeric-looking string vs. actual number: use quotes to force string type
# set NUMERIC_CODES := "100" "200" "300" ;   # strings
# set NUMERIC_SET   := 100 200 300 ;         # numbers -- distinct from the above

#### The `symbolic` Keyword on Parameters

A standard AMPL parameter is numeric: it holds a floating-point number. To allow a parameter
to hold a string value — for example, to designate which member of a set plays a special role —
the declaration must include the keyword `symbolic`.

A symbolic parameter's value may be any string or number, just as a set member may be either.
String values of a symbolic parameter **may not participate in arithmetic**. The numeric
comparison operators (`<`, `<=`, `>`, `>=`) are not permitted as attributes of a symbolic
parameter; only `=` (definition), `default`, and `in` (membership constraint) are available.

The `in SETNAME` attribute on a symbolic parameter declares that the parameter's value must
be a member of `SETNAME`. AMPL validates this constraint when data is loaded; supplying a
value not in `SETNAME` produces an error.

Two symbolic parameters may be constrained to differ using `<> other_param`. This is
validated at data-load time.

**Primary uses of symbolic parameters:**
1. **Designating a special member** of a set (e.g., an entry node, a reference product).
2. **Associating descriptive text** with each member of an index set (e.g., full city names).
3. **Storing categorical labels** that drive conditional logic in model expressions.

In [ ]:
# Symbolic parameter syntax variants

# Scalar symbolic parameter — value is one member of SETNAME
set INTER;                              # set of intersections in a road network
param entr symbolic in INTER;          # entrance: must be a member of INTER
param exit symbolic in INTER, <> entr; # exit: must be in INTER and differ from entr

# Indexed symbolic parameter — one string value per set member
set ORIG;                              # short origin codes (e.g., GARY, CLEV, PITT)
param orig_name {ORIG} symbolic;       # full descriptive name for each origin

# Symbolic parameter with a default
param phase symbolic, default "I";     # algorithm phase; defaults to "I" if no data given

#### Example 1 — Minimal String Set Declaration and Use

This example shows the minimal pattern: declare a set, reference it as an index domain,
and supply its members in a data section. The set members are simple alphanumeric strings
that require no quoting in data context.

In [ ]:
# --- Model ---
set PROD;                              # products; members are strings
param rate   {PROD} >= 0;             # production rate (tons/hour)
param profit {PROD};                  # profit per ton
param market {PROD} >= 0;             # market upper bound

var Make {p in PROD} >= 0, <= market[p];

maximize TotalProfit:
    sum {p in PROD} profit[p] * Make[p];

# --- Data ---
data;
set PROD := bands coils plate ;

param rate   := bands 200  coils 140  plate 160 ;
param profit := bands  25  coils  30  plate  29 ;
param market := bands 6000 coils 4000 plate 3500 ;

#### Example 2 — Symbolic Parameters with a String Set (Network Flow)

This is the canonical use of `symbolic`: two parameters designate special members of a
string set. The `in SETNAME` attribute enforces membership; `<> entr` enforces distinctness.
Both constraints are checked when data is loaded, not at solve time.

This example is drawn directly from the AMPL book (Section 15.2, `netmax.mod`).

In [ ]:
# Maximum-flow network model — symbolic parameters designate source and sink

set INTER;                                        # intersections (string identifiers)

param entr symbolic in INTER;                     # entrance node: must be a member of INTER
param exit symbolic in INTER, <> entr;            # exit node: must be in INTER, differ from entr

# The symbolic parameters are used directly as set expressions in the within clause
set ROADS within (INTER diff {exit}) cross (INTER diff {entr});

param cap {ROADS} >= 0;                           # road capacity
var Traff {(i,j) in ROADS} >= 0, <= cap[i,j];    # traffic flow on each road

maximize Entering_Traff:
    sum {(entr,j) in ROADS} Traff[entr,j];

subject to Balance {k in INTER diff {entr, exit}}:
    sum {(i,k) in ROADS} Traff[i,k] = sum {(k,j) in ROADS} Traff[k,j];

# --- Data ---
data;

set INTER := a b c d e f g ;      # unquoted: all alphanumeric

param entr := a ;                 # string value assigned to symbolic parameter
param exit := g ;                 # must differ from entr; both must be in INTER

param: ROADS: cap :=
    a b  50   a c 100
    b d  40   b e  20
    c d  60   c f  20
    d e  50   d f  60
    e g  70   f g  70 ;

#### Example 3 — Indexed Symbolic Parameter as Descriptive Label

Short string codes serve as set members (they are compact and easy to write as dummy
indices). Longer, human-readable descriptions are stored in an indexed symbolic parameter.
The descriptions play no role in the optimization; they are available for reporting via
`display` or `printf`.

In [ ]:
# Transportation model — short codes as set members, descriptive names as symbolic parameters

set ORIG;                              # origin codes (short strings)
set DEST;                              # destination codes (short strings)

param orig_name {ORIG} symbolic;       # full city name for each origin
param dest_name {DEST} symbolic;       # full city name for each destination

param supply {ORIG} >= 0;
param demand {DEST} >= 0;
param cost   {ORIG, DEST} >= 0;

var Trans {ORIG, DEST} >= 0;

minimize TotalCost:
    sum {i in ORIG, j in DEST} cost[i,j] * Trans[i,j];

subject to Supply {i in ORIG}:
    sum {j in DEST} Trans[i,j] = supply[i];

subject to Demand {j in DEST}:
    sum {i in ORIG} Trans[i,j] = demand[j];

# --- Data ---
data;

set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN WIN STL FRE LAF ;

param orig_name :=           # long strings require quotes
    GARY  "Gary, Indiana"
    CLEV  "Cleveland, Ohio"
    PITT  "Pittsburgh, Pennsylvania" ;

param dest_name :=
    FRA   "Frankfurt"
    DET   "Detroit"
    LAN   "Lansing"
    WIN   "Windsor"
    STL   "St. Louis"
    FRE   "Fresno"
    LAF   "Lafayette" ;

param supply := GARY 1400  CLEV 2600  PITT 2900 ;
param demand := FRA 900  DET 1200  LAN 600  WIN 400  STL 1700  FRE 1100  LAF 1000 ;

# Report: print full names alongside codes
# (This printf statement is for illustration; it would appear in a script)
# printf {i in ORIG} "%s (%s)\n", orig_name[i], i;

#### Membership Tests with String Literals

The `in` operator tests whether an expression evaluates to a member of a set. When the
left operand is a string literal in **model context**, quotes are mandatory. The result is
a Boolean expression (1 if true, 0 if false) and may appear in:
- Conditional expressions (`if ... then ... else ...`)
- Indexing filter conditions (`{p in PROD: ... in OTHER_SET}`)
- `check` statements
- Parameter definitions

The `not in` form is also available and returns the logical negation.

In [ ]:
set NUTR;                              # all nutrients
set MINREQ within NUTR;                # nutrients with a minimum requirement

# Membership test: is the string "B2" a member of NUTR?
# Quotes are required in model context.
param has_B2 = if "B2" in NUTR then 1 else 0;

# Filtering: products not in the MINREQ set
# (NUTR and MINREQ are sets; 'n' ranges over NUTR)
set UNRESTRICTED = {n in NUTR: n not in MINREQ};

# Check statement: the string "CAL" must be a nutrient
check "CAL" in NUTR;

# Subset test: MINREQ must be a subset of NUTR
check MINREQ within NUTR;

#### Set Operators on String Sets

All five binary set operators — `union`, `inter`, `diff`, `symdiff`, `cross` — work
identically on string sets as on numeric sets. There is no type-specific behavior: the
operators compare members by value (using AMPL's equality semantics for strings, which is
case-sensitive) and produce new unordered sets.

One consequence of case sensitivity: `"Gary"` and `"GARY"` are distinct members, so
`{"gary"} inter {"GARY"}` is the empty set.

The singleton-set wrapper `{"string"}` is required whenever a single string literal must be
used as a set operand. Writing `ORIG union "CLEV"` is a type error; the correct form is
`ORIG union {"CLEV"}`.

In [ ]:
# Set operators on string sets

set FOODS_A = {"bread", "milk", "cheese"};
set FOODS_B = {"milk", "cheese", "eggs", "butter"};

set ALL_FOODS   = FOODS_A union  FOODS_B;   # {"bread","milk","cheese","eggs","butter"}
set COMMON      = FOODS_A inter  FOODS_B;   # {"milk","cheese"}
set ONLY_A      = FOODS_A diff   FOODS_B;   # {"bread"}
set EXCLUSIVE   = FOODS_A symdiff FOODS_B;  # {"bread","eggs","butter"}

display card(ALL_FOODS);    # 5
display card(COMMON);       # 2
display card(ONLY_A);       # 1
display card(EXCLUSIVE);    # 3

# Case sensitivity: "Milk" != "milk"
set FOODS_C = {"Milk", "Cheese"};           # different capitalisation
set OVERLAP = FOODS_A inter FOODS_C;        # {} — empty; case mismatch
display card(OVERLAP);      # 0

# Singleton set required when using a string literal as a set operand
set PROD;
set PROD_EXCL_BANDS = PROD diff {"bands"};  # correct: {"bands"} is a singleton set
# PROD diff "bands" would be a type error

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use short alphanumeric strings as set members (e.g., <code>GARY</code>, <code>bands</code>,
<code>B2</code>). Short codes serve directly as dummy indices in indexing expressions and
require no quoting in data files. Store longer, human-readable descriptions in an indexed
<code>symbolic</code> parameter (e.g., <code>param orig_name {ORIG} symbolic;</code>) so they
are available for reporting without cluttering the model's index domains.</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
When a scalar <code>symbolic</code> parameter designates a special member of a set, always
pair it with the <code>in SETNAME</code> attribute. This turns a latent data error (assigning
a non-existent node as the entrance) into an immediate, informative diagnostic at data-load
time rather than a silent wrong answer at solve time.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
A string literal in <strong>model context</strong> (inside <code>.mod</code> files and
<code>ampl.eval()</code> calls) must always be quoted, even if it consists only of letters.
Writing <code>"Day" in SHIFTS</code> is correct; writing <code>Day in SHIFTS</code> is a
reference to a declared set named <code>Day</code>, not to the string <code>"Day"</code>.
The error may be silent if a set named <code>Day</code> happens to exist.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The number <code>1</code> and the string <code>"1"</code> are distinct AMPL values. A set
containing both has cardinality 2. If numeric string codes are required (e.g., order IDs
like <code>"100"</code>, <code>"200"</code>), supply them with quotes in the data file to
guarantee they are stored as strings. Supplying them without quotes creates numeric set
members, which may cause an error when they are used in a context expecting a string
(e.g., as a value for a <code>symbolic</code> parameter that is subsequently passed to a
string function).</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
A <code>symbolic</code> parameter's value <strong>cannot participate in arithmetic</strong>.
Declaring <code>param phase symbolic;</code> and then writing <code>phase + 1</code> in a
model expression is an error that AMPL reports at model-generation time. If a parameter
needs both string labels (for reporting) and numeric values (for arithmetic), they must be
two separate parameters: one <code>symbolic</code> for the label and one numeric for the
computation.</div>

#### amplpy Equivalents

In amplpy, string sets are assigned from Python lists or pandas Series of strings. The
Python objects do not need any quoting — Python strings are passed to AMPL directly.
Case is preserved exactly.

Symbolic parameters are assigned using the same `set_values` or property-setter API as
numeric parameters, with the values as Python strings.

| Operation | amplpy pattern | Notes |
|---|---|---|
| Assign a string set from a list | `ampl.set["PROD"] = ["bands", "coils", "plate"]` | Must declare `set PROD;` first |
| Assign via `set_values` | `ampl.get_set("PROD").set_values(["bands", "coils"])` | Equivalent to property assignment |
| Assign from a pandas Series | `ampl.get_set("PROD").set_values(df["prod"].tolist())` | Convert to Python list first |
| Assign a scalar `symbolic` param | `ampl.param["entr"] = "a"` | Calls `set()` on the Parameter |
| Assign an indexed `symbolic` param | `ampl.get_parameter("orig_name").set_values({"GARY": "Gary, Indiana", ...})` | Dict keyed by set member |
| Read string set members back | `ampl.get_set("PROD").members()` | Returns a Python list of strings |
| Read symbolic param value | `ampl.param["entr"].value()` | Returns a Python `str` for string-valued params |

In [ ]:
from amplpy import AMPL
import pandas as pd

ampl = AMPL()

# Define model structure
ampl.eval("""
    set PROD;
    set ORIG;
    param orig_name {ORIG} symbolic;
    param entr symbolic in ORIG;
""")

# Pattern 1: assign string set via property setter
ampl.set["PROD"] = ["bands", "coils", "plate"]

# Pattern 2: assign string set via set_values
ampl.get_set("ORIG").set_values(["GARY", "CLEV", "PITT"])

# Pattern 3: assign string set from a pandas Series
df = pd.DataFrame({"prod": ["bands", "coils", "plate"]})
ampl.get_set("PROD").set_values(df["prod"].tolist())

# Assign indexed symbolic parameter from a dict
ampl.get_parameter("orig_name").set_values({
    "GARY": "Gary, Indiana",
    "CLEV": "Cleveland, Ohio",
    "PITT": "Pittsburgh, Pennsylvania"
})

# Assign scalar symbolic parameter
ampl.param["entr"] = "GARY"              # Python str; value must be a member of ORIG

# Read back: string set members
prod_members = ampl.get_set("PROD").members()   # returns a Python list of strings
print(prod_members)                              # ['bands', 'coils', 'plate']

# Read back: scalar symbolic parameter value
entrance = ampl.param["entr"].value()            # returns Python str 'GARY'
print(entrance)

#### Edge Cases

**Edge case 1: Numbers and strings that look alike — type confusion in mixed data.**

AMPL stores set members either as floating-point numbers or as strings, never as both.
The number `100` and the string `"100"` are distinct objects with distinct types. If a data
file supplies `100` (without quotes) for a set that is referenced in a context expecting
a string — for example, as the value of a `symbolic` parameter — AMPL may raise a type
error or silently coerce. The safe rule is: use quotes for every member that is
conceptually a label rather than a quantity.

In [ ]:
# Demonstrating number vs. string distinction

# These two sets have different member types:
set ORDER_IDS_NUM = {100, 200, 300};      # numeric members
set ORDER_IDS_STR = {"100", "200", "300"};# string members

display card(ORDER_IDS_NUM);  # 3
display card(ORDER_IDS_STR);  # 3

# Intersection is empty: number 100 != string "100"
display ORDER_IDS_NUM inter ORDER_IDS_STR;  # {}

# In data context: unquoted integer literals produce numeric members
# Quoted integer strings produce string members
# Mixing both in one set is valid but almost always a modelling error:
# set MIXED = {100, "100"};  # two distinct members; cardinality = 2

# Safe pattern for order-ID style codes: always quote in data file
set ORDERS;
data;
set ORDERS := "100" "200" "300" ;   # stored as strings, not numbers

**Edge case 2: Case sensitivity in membership tests and set operations.**

Because AMPL is case-sensitive for strings, `union`, `inter`, and `diff` treat
`"Gary"` and `"GARY"` as unrelated members. A common mistake arises when data comes
from two sources with inconsistent capitalisation. The intersection of such sets will be
empty even though the sets appear to overlap, and a `within` constraint will fail.

In [ ]:
# Case-sensitivity edge case

# Two sets from different sources; capitalisation differs
set SOURCE_A = {"Gary", "Cleveland", "Pittsburgh"};   # title-cased
set SOURCE_B = {"GARY", "CLEV", "PITT"};              # all-caps abbreviations

display SOURCE_A inter SOURCE_B;    # {} — empty; no member matches any other
display card(SOURCE_A union SOURCE_B);  # 6 — all six treated as distinct

# A within check catches the mismatch at data-load time:
# set DECLARED_ORIG = {"GARY", "CLEV", "PITT"};
# set LINKS_ORIG within DECLARED_ORIG;
# data;
# set LINKS_ORIG := Gary ;    # ERROR: 'Gary' not in DECLARED_ORIG (case mismatch)

# In amplpy: Python strings must match AMPL strings exactly (case-sensitive)
# ampl.set["ORIG"] = ["Gary"]    # will not match a set defined with 'GARY'
# ampl.set["ORIG"] = ["GARY"]    # correct

**Edge case 3: Symbolic parameter value not in the `in` set — error at data-load time, not solve time.**

The `in SETNAME` attribute on a `symbolic` parameter is enforced when data is loaded, before
any constraint generation or solve. The error message identifies the parameter and the
invalid value. This early detection is one of the main reasons to declare `in` on symbolic
parameters: it catches bad instance data before the solver is invoked.

In [ ]:
# Symbolic parameter 'in' constraint — enforced at data-load time

set INTER;
param entr symbolic in INTER;
param exit symbolic in INTER, <> entr;

data;
set INTER := a b c d e f g ;
param entr := a ;
param exit := g ;   # valid: g is in INTER and g != a

# If exit were assigned a value not in INTER, AMPL would report an error here:
# param exit := z ;   # ERROR at data-load: z is not a member of INTER

# If both params were assigned the same value, AMPL would report an inequality violation:
# param exit := a ;   # ERROR at data-load: exit must differ from entr

# The model itself is never reached in the error cases above.
# No constraint or variable is generated; the session reports the violation immediately.

**Edge case 4: Using a string set member directly in a set expression.**

A `symbolic` parameter whose value is a string can be used inside a set expression as a
singleton reference. In the network-flow model, `{exit}` and `{entr}` construct singleton
sets from the symbolic parameters, which are then passed to `diff`. This is correct:
the braces `{expr}` construct a singleton set from the value of `expr`, whether `expr` is
a string literal or a symbolic parameter. The syntax `{exit}` and `{"g"}` produce the
same set when `exit` has the value `"g"`.

In [ ]:
# Using a symbolic parameter as a singleton in a set expression

set INTER;
param entr symbolic in INTER;
param exit symbolic in INTER, <> entr;

# {exit} constructs a singleton set {value-of-exit}
# INTER diff {exit} removes the exit node from the set of all nodes
set NON_EXIT   = INTER diff {exit};   # all nodes except the exit
set NON_ENTRY  = INTER diff {entr};   # all nodes except the entry
set INTERNAL   = INTER diff {entr, exit}; # interior nodes only

# These sets update automatically when entr or exit data changes
# (symbolic definitions re-evaluate when dependencies change)

# With data:
data;
set INTER := a b c d e f g ;
param entr := a ;
param exit := g ;

display NON_EXIT;    # {a, b, c, d, e, f}
display INTERNAL;    # {b, c, d, e, f}

#### Cross-References

See also:
- [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — `=`, `default`, and `:=` for set definitions
- [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — `union`, `inter`, `diff`, `symdiff`, `cross` in depth
- [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — properties of unordered sets, valid functions, compound sets